# Heliyon Revision Experiments — HELIYON-D-26-02293

New analyses requested by reviewers for *"Does Cross-Exchange Fragmentation Distort
Cryptocurrency Risk Measures?"*

| Section | Analysis | Reviewer comment |
|---|---|---|
| A0 | Corrected full-sample variance shares (Table 2 recomputation) | internal consistency |
| A | Stress-conditional fragmentation (variance shares under high-vol / stress / anomaly regimes) | R3 #2 |
| B | Newey-West significance tests for mean cross-venue spreads (Table 1) | R3 #8 |
| C | Rolling-window robustness, w ∈ {14, 30, 60} | R3 #4 |
| D1 | Micro-depeg threshold sweep τ ∈ {5, 10, 15, 20} bps (expanding-window logistic) | R2 #5, R3 #3 |
| D2 | Moving-block bootstrap block-length robustness L ∈ {7, 14, 21} | R3 #6 |
| E | Economic context of the 10 bp threshold (deviation distribution) | R2 #3 |

**How to run:** Runtime → Run all. The first cell asks you to upload `colab_data.zip`.
The last cell downloads `heliyon_v2_outputs.zip` with all result CSVs.


In [ ]:
# --- Setup: obtain data (upload colab_data.zip when prompted on Colab) ---
import os, zipfile

IN_COLAB = False
try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if not os.path.isdir("data"):
    if not os.path.exists("colab_data.zip"):
        assert IN_COLAB, "colab_data.zip not found next to this notebook"
        print(">>> Please upload colab_data.zip")
        from google.colab import files
        files.upload()
    with zipfile.ZipFile("colab_data.zip") as z:
        z.extractall("data")

os.makedirs("outputs", exist_ok=True)
print("Data files:", sorted(os.listdir("data")))


In [ ]:
# --- Imports and global settings ---
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 42
VENUES = ["binance", "kraken", "coinbase"]
PAIRS = [("binance", "kraken", "bk"), ("binance", "coinbase", "bc"), ("kraken", "coinbase", "kc")]

panel = pd.read_csv("data/panel_overlap_btc_eth_3venues.csv", parse_dates=["date_utc"])
spreads = pd.read_csv("data/spreads_overlap_btc_eth_3venues.csv", parse_dates=["date_utc"])
usdt = (pd.read_csv("data/usdt_kraken.csv", parse_dates=["date_utc"])
        .rename(columns={"date_utc": "date"})
        [["date", "close_px", "volume_base"]]
        .sort_values("date").reset_index(drop=True))
depeg10 = pd.read_csv("data/usdt_depeg_dataset.csv", parse_dates=["date"])

print(f"panel: {panel.shape}, spreads: {spreads.shape}, usdt: {usdt.shape}, depeg10: {depeg10.shape}")


## Section A0 — Corrected Table 2: full-sample variance shares

Recomputation of the manuscript's Table 2 under the exact Eq. (6) definition,
Var(Δ)/Var(r), on the final cleaned three-venue panel: all six ordered venue pairs
per asset; report mean and max share (in percent).


In [ ]:
# --- A0. Corrected full-sample variance shares (Table 2) ---
from itertools import permutations

rows = []
for asset, g in panel.groupby("asset"):
    g = g.sort_values("date_utc").copy()
    for v in VENUES:
        g[f"ret_{v}"] = np.log(g[f"close_{v}"]).diff()
    g = g.dropna()
    shares = []
    for i, j in permutations(VENUES, 2):
        delta = g[f"ret_{i}"] - g[f"ret_{j}"]
        shares.append(100 * delta.var() / g[f"ret_{i}"].var())
    rows.append({"asset": asset, "n_days": len(g),
                 "mean_var_share_pct": float(np.mean(shares)),
                 "max_var_share_pct": float(np.max(shares))})

table2 = pd.DataFrame(rows)
table2.to_csv("outputs/variance_share_table2.csv", index=False)
print(table2.round(4).to_string(index=False))


## Section A — Stress-conditional fragmentation (R3 #2)

Reviewer 3 notes that near-perfect daily integration for BTC/ETH is unsurprising and asks us
to investigate settings where fragmentation is *expected* to matter. We recompute the
variance share Var(Δ)/Var(r) and spread magnitudes conditional on market states in which
limits-to-arbitrage theory predicts wider dislocations: elevated volatility regimes,
extreme negative return days, and identified anomaly days.


In [ ]:
# --- A. Stress-conditional variance decomposition and spread magnitudes ---
rows = []
for asset, g in panel.groupby("asset"):
    g = g.sort_values("date_utc").copy()
    for v in VENUES:
        g[f"ret_{v}"] = np.log(g[f"close_{v}"]).diff()
    g["ret_avg"] = g[[f"ret_{v}" for v in VENUES]].mean(axis=1)
    g["rv30"] = g["ret_avg"].rolling(30).std()

    s = spreads.loc[spreads.asset == asset,
                    ["date_utc", "spread_rel_bk", "spread_rel_bc", "spread_rel_kc"]]
    g = g.merge(s, on="date_utc", how="left")
    for _, _, tag in PAIRS:
        col = f"spread_rel_{tag}"
        mu = g[col].rolling(30).mean()
        sd = g[col].rolling(30).std()
        g[f"z_{tag}"] = (g[col] - mu) / sd

    g = g.dropna().reset_index(drop=True)

    conditions = {
        "Full sample": pd.Series(True, index=g.index),
        "High-vol regime (top quartile 30d RV)": g["rv30"] >= g["rv30"].quantile(0.75),
        "High-vol regime (top decile 30d RV)": g["rv30"] >= g["rv30"].quantile(0.90),
        "Stress days (worst decile returns)": g["ret_avg"] <= g["ret_avg"].quantile(0.10),
        "Anomaly days (|Z|>=3, any pair)": (g[[f"z_{t}" for _, _, t in PAIRS]].abs() >= 3).any(axis=1),
    }
    for cond_name, mask in conditions.items():
        sub = g[mask]
        for vi, vj, tag in PAIRS:
            delta = sub[f"ret_{vi}"] - sub[f"ret_{vj}"]
            var_share = delta.var() / sub[f"ret_{vi}"].var()
            abs_spr = sub[f"spread_rel_{tag}"].abs()
            rows.append({
                "asset": asset, "condition": cond_name, "pair": tag.upper(),
                "n_days": int(len(sub)),
                "var_share_pct": 100 * var_share,
                "mean_abs_spread_bps": 1e4 * abs_spr.mean(),
                "p95_abs_spread_bps": 1e4 * abs_spr.quantile(0.95),
                "max_abs_spread_bps": 1e4 * abs_spr.max(),
            })

condA = pd.DataFrame(rows)
condA.to_csv("outputs/stress_conditional_fragmentation.csv", index=False)

# Compact summary for the paper table: aggregate across the three pairs
summ = (condA.groupby(["asset", "condition"])
        .agg(n_days=("n_days", "first"),
             mean_var_share_pct=("var_share_pct", "mean"),
             max_var_share_pct=("var_share_pct", "max"),
             mean_abs_spread_bps=("mean_abs_spread_bps", "mean"),
             p95_abs_spread_bps=("p95_abs_spread_bps", "mean"))
        .reset_index())
summ.to_csv("outputs/stress_conditional_summary.csv", index=False)
print(summ.round(3).to_string(index=False))


## Section B — Statistical significance of mean spreads for Table 1 (R3 #8)

Newey-West (HAC) t-tests of H0: mean relative spread = 0, per asset and venue pair.


In [ ]:
# --- B. Newey-West significance of mean cross-venue spreads ---
rows = []
for asset, s in spreads.groupby("asset"):
    s = s.sort_values("date_utc")
    for _, _, tag in PAIRS:
        y = s[f"spread_rel_{tag}"].dropna().values
        n = len(y)
        hac_lags = int(np.floor(4 * (n / 100) ** (2 / 9)))  # Newey-West rule of thumb
        fit = sm.OLS(y, np.ones((n, 1))).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
        rows.append({
            "asset": asset, "pair": tag.upper(), "n": n,
            "mean_spread": y.mean(), "std_spread": y.std(ddof=1),
            "t_NW": fit.tvalues[0], "p_NW": fit.pvalues[0], "hac_lags": hac_lags,
        })

sigB = pd.DataFrame(rows)
sigB.to_csv("outputs/spread_significance.csv", index=False)
print(sigB.round(6).to_string(index=False))


## Section C — Rolling-window robustness w ∈ {14, 30, 60} (R3 #4)

Anomaly frequency (share of days with |Z| ≥ 3) for each window length. The variance
decomposition itself does not depend on w; only the standardized-spread diagnostic does.


In [ ]:
# --- C. Z-score window robustness ---
rows = []
for asset, s in spreads.groupby("asset"):
    s = s.sort_values("date_utc").copy()
    for w in [14, 30, 60]:
        freqs = []
        for _, _, tag in PAIRS:
            col = f"spread_rel_{tag}"
            mu = s[col].rolling(w).mean()
            sd = s[col].rolling(w).std()
            z = ((s[col] - mu) / sd).dropna()
            freq = (z.abs() >= 3).mean()
            freqs.append(freq)
            rows.append({"asset": asset, "window": w, "pair": tag.upper(),
                         "n_valid": int(len(z)), "anomaly_freq_pct": 100 * freq})
        rows.append({"asset": asset, "window": w, "pair": "ALL(avg)",
                     "n_valid": int(len(z)), "anomaly_freq_pct": 100 * float(np.mean(freqs))})

winC = pd.DataFrame(rows)
winC.to_csv("outputs/window_robustness.csv", index=False)
print(winC[winC.pair == "ALL(avg)"].round(3).to_string(index=False))


## Section D1 — Threshold sweep τ ∈ {5, 10, 15, 20} bps (R2 #5, R3 #3)

True expanding-window walk-forward logistic regression (re-estimated at every forecast
origin, initial window = first 70%), identical to the manuscript protocol. Adds τ = 5 bps
below the 10 bps baseline. The 10 bps row must reproduce the manuscript numbers
(AUROC ≈ 0.832, AUPRC ≈ 0.422).


In [ ]:
# --- D1. Expanding-window threshold sweep ---
FEATURE_COLS = [
    "ret_lag1", "ret_lag2", "ret_lag3",
    "ret_vol_7", "ret_vol_14", "ret_vol_30",
    "volume_lag1", "volume_lag3",
    "volume_vol_7", "volume_vol_14",
]

def build_depeg_dataset(usdt_df, thresh):
    df = usdt_df.copy()
    df["abs_dev"] = (df["close_px"] - 1.0).abs()
    df["is_offpeg"] = (df["abs_dev"] > thresh).astype(int)
    df["depeg"] = df["is_offpeg"].shift(-1)
    df["ret"] = np.log(df["close_px"]).diff()
    for k in [1, 2, 3]:
        df[f"ret_lag{k}"] = df["ret"].shift(k)
    for w in [7, 14, 30]:
        df[f"ret_vol_{w}"] = df["ret"].rolling(w).std()
    df["volume_lag1"] = df["volume_base"].shift(1)
    df["volume_lag3"] = df["volume_base"].shift(3)
    for w in [7, 14]:
        df[f"volume_vol_{w}"] = df["volume_base"].rolling(w).std()
    return df.dropna().reset_index(drop=True)

def expanding_window_logistic(X, y, T0):
    preds = np.zeros(len(y) - T0)
    for t in range(T0, len(y)):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X[:t])
        X_pt = scaler.transform(X[t:t + 1])
        clf = LogisticRegression(max_iter=2000, random_state=SEED)
        clf.fit(X_tr, y[:t])
        preds[t - T0] = clf.predict_proba(X_pt)[0, 1]
    return preds

def safe_metric(y_true, probs, fn):
    return fn(y_true, probs) if len(set(y_true)) > 1 else float("nan")

rows = []
for thresh in [0.0005, 0.0010, 0.0015, 0.0020]:
    df = build_depeg_dataset(usdt, thresh)
    X = df[FEATURE_COLS].values
    y = df["depeg"].astype(int).values
    T0 = int(len(y) * 0.70)
    preds = expanding_window_logistic(X, y, T0)
    y_test = y[T0:]
    rows.append({
        "threshold_bps": int(round(thresh * 1e4)),
        "n": len(y), "event_rate": y.mean(),
        "events_total": int(y.sum()), "events_test": int(y_test.sum()),
        "AUROC": safe_metric(y_test, preds, roc_auc_score),
        "AUPRC": safe_metric(y_test, preds, average_precision_score),
        "Brier": brier_score_loss(y_test, preds),
    })
    print(f"tau={rows[-1]['threshold_bps']:>2} bps  event_rate={y.mean():.3f}  "
          f"AUROC={rows[-1]['AUROC']:.3f}  AUPRC={rows[-1]['AUPRC']:.3f}  Brier={rows[-1]['Brier']:.3f}")

sweepD = pd.DataFrame(rows)
sweepD.to_csv("outputs/threshold_robustness_extended.csv", index=False)


## Section D2 — Block-length robustness L ∈ {7, 14, 21} (R3 #6)

Moving-block bootstrap (B = 500, seed 42) for the logistic coefficients at the 10 bps
threshold, re-run with shorter and longer blocks than the baseline L = 14.


In [ ]:
# --- D2. Moving-block bootstrap with alternative block lengths ---
def moving_block_indices(n, block_len, rng):
    if block_len <= 1:
        return rng.integers(0, n, size=n)
    starts = rng.integers(0, n - block_len + 1, size=int(np.ceil(n / block_len)))
    idx = []
    for s in starts:
        idx.extend(range(s, s + block_len))
    return np.array(idx[:n], dtype=int)

def fit_logit_coefs(X, y):
    Xs = StandardScaler().fit_transform(X)
    m = LogisticRegression(max_iter=4000)
    m.fit(Xs, y)
    return m.coef_[0].copy()

df10 = build_depeg_dataset(usdt, 0.001)
split = int(len(df10) * 0.70)
train = df10.iloc[:split]
X_train = train[FEATURE_COLS]
y_train = train["depeg"].astype(int)
point = fit_logit_coefs(X_train, y_train)

B = 500
rows = []
for L in [7, 14, 21]:
    rng = np.random.default_rng(SEED)
    boot, failed = [], 0
    n = len(train)
    for b in range(B):
        idx = moving_block_indices(n, L, rng)
        yb = y_train.iloc[idx]
        if yb.nunique() < 2:
            failed += 1
            continue
        try:
            boot.append(fit_logit_coefs(X_train.iloc[idx], yb))
        except Exception:
            failed += 1
    boot = np.array(boot)
    ci_lo = np.percentile(boot, 2.5, axis=0)
    ci_hi = np.percentile(boot, 97.5, axis=0)
    for j, feat in enumerate(FEATURE_COLS):
        p = 2 * min((boot[:, j] <= 0).mean(), (boot[:, j] >= 0).mean())
        rows.append({"block_len": L, "feature": feat, "coef_point": point[j],
                     "ci95_lo": ci_lo[j], "ci95_hi": ci_hi[j], "p_boot": p,
                     "B_effective": len(boot)})
    print(f"L={L}: {len(boot)}/{B} effective draws")

blockD = pd.DataFrame(rows)
blockD.to_csv("outputs/blocklen_robustness.csv", index=False)

key_feats = ["ret_vol_30", "ret_lag1", "ret_lag3", "volume_lag3"]
print(blockD[blockD.feature.isin(key_feats)]
      .pivot(index="feature", columns="block_len", values="p_boot").round(3))


## Section E — Economic context of the 10 bp threshold (R2 #3)

Where does 10 bp sit in the USDT/USD deviation distribution, and how does it compare
with price granularity and typical deviations?


In [ ]:
# --- E. Deviation distribution context for the threshold choice ---
dev_bps = 1e4 * (usdt["close_px"] - 1.0).abs()

ctx = {
    "n_days": int(len(dev_bps)),
    "mean_dev_bps": dev_bps.mean(),
    "median_dev_bps": dev_bps.median(),
    "p75_dev_bps": dev_bps.quantile(0.75),
    "p90_dev_bps": dev_bps.quantile(0.90),
    "p95_dev_bps": dev_bps.quantile(0.95),
    "max_dev_bps": dev_bps.max(),
    "pctile_rank_of_10bps": 100 * (dev_bps <= 10).mean(),
    "share_days_ge_5bps_pct": 100 * (dev_bps >= 5).mean(),
    "share_days_ge_10bps_pct": 100 * (dev_bps >= 10).mean(),
    "share_days_ge_15bps_pct": 100 * (dev_bps >= 15).mean(),
    "share_days_ge_20bps_pct": 100 * (dev_bps >= 20).mean(),
    "ratio_10bps_to_mean": 10 / dev_bps.mean(),
    "ratio_10bps_to_median": 10 / dev_bps.median(),
}
ctxE = pd.DataFrame([ctx])
ctxE.to_csv("outputs/depeg_threshold_context.csv", index=False)
print(ctxE.T.rename(columns={0: "value"}).round(3))


## Package outputs

In [ ]:
# --- Package all outputs and download ---
import shutil
shutil.make_archive("heliyon_v2_outputs", "zip", "outputs")
print("Created heliyon_v2_outputs.zip with:", sorted(os.listdir("outputs")))
if IN_COLAB:
    from google.colab import files
    files.download("heliyon_v2_outputs.zip")
